In [9]:
from zhinst.toolkit import Session, Sequence, Waveforms
from zhinst.toolkit.waveform import Wave, OutputType
import numpy as np
import time

In [10]:
def comp_upload_seq(awgnode,code,device,session):

    awg_node = device.awgs[awgnode]
    awg = session.modules.awg
    awg.device(device.serial)
    awg.index(awgnode)
    awg.sequencertype('sg' if device.device_type.startswith('SHFSG') or device.device_type.startswith('SHFQC') else 'auto-detect')
    awg.execute()
    awg.compiler.sourcestring(code)

    # The following lines are not mandatory but only to ensure that everything was compiled and uploaded correctly. 
    timeout = 100.0  # seconds
    compiler_status = awg.compiler.status()
    start = time.time()
    while compiler_status == -1:
        if time.time() - start >= timeout:
            raise TimeoutError("Program compilation timed out")
        time.sleep(0.1)
        compiler_status = awg.compiler.status()
        if compiler_status == 1:
            print(awg.compiler.statusstring())
            raise RuntimeError(
                "Error during sequencer compilation. Check the log for detailed information"
            )
        if compiler_status == 2:
            print(f"Warning during sequencer compilation {awg.compiler.statusstring()}")
    # Check and wait until the elf upload to the device was successful
    print("Sequence loaded.")
    progress = awg.progress()
    while progress < 1.0 or awg.elf.status() == 2 or awg_node.ready() == 0:
        if time.time() - start >= timeout:
            raise TimeoutError(f"Program upload timed out")
        time.sleep(0.1)
        progress = awg.progress()
    if awg.elf.status() or not awg_node.ready():
        raise RuntimeError(
            "Error during upload of ELF file. Check the log for detailed information"
        )
    print("AWG ready for data upload.")

    return

def add_truncgauss_risefall(wave,risefall_t,sclk):
    '''
    Given a waveform with unit amplitude and a rise-fall time, this returns the same wave with 
    a truncated gaussian envelope with a flat top.
    '''
    def trunc_gauss(x,tau,sigma):
        return ((np.exp(-0.5*((x-tau/2)/sigma)**2) - np.exp(-0.5*((tau/2)/sigma)**2))/(1-np.exp(-0.5*((tau/2)/sigma)**2)))

    tau = int(risefall_t*sclk)*2 #rise + fall time in points
    if (tau) > wave.size : raise Exception("Wave too short for complete rise and fall.")
    sigma = tau/4 # of the gaussian
    #calculate envelope
    truncgaussarr = np.array([trunc_gauss(i,tau,sigma) for i in range(tau)])
    #apply envelope
    wavef = np.copy(wave)
    wavef[:int(tau/2)] = np.multiply(wave[:int(tau/2)],truncgaussarr[:int(tau/2)])
    wavef[-int(tau/2):] = np.multiply(wave[-int(tau/2):],truncgaussarr[-int(tau/2):])
    return wavef

def gen_sin(freq,phase,amp,len,sclk):
    '''
    Generate a sin wave with frequency in Hz, phase in radians, length in seconds, sclk in samples/sec
    '''
    no_pts = int(len*sclk)
    omega_pt = freq/sclk
    wave = amp*np.sin([(omega_pt*i+phase) for i in range(no_pts)])
    return wave

def round_off_wvf_to_16_wvfpoints(wavf,front=False,back=False):
    if wavf.size <= 32 : return np.hstack((wavf,np.zeros(32-wavf.size)))
    if back == True:
        return np.hstack((wavf,np.zeros(16-int(wavf.size%16))))
    if front == True:
        return np.hstack((np.zeros(16-int(wavf.size%16)),wavf))



In [11]:
session = Session("localhost")
session.devices.visible()

['dev8099']

In [12]:
device = session.connect_device("dev8099")

In [13]:
OUT_CHANNEL = 1
AWG_CHANNEL = 0

device.system.awg.channelgrouping(2)   #on 4x1 mode
device.triggers.out['*'].source(8)
device.sigouts['*'].on(True)
device.sigouts["*"].range(1)
device.awgs[0].outputs["*"].amplitude(1.0)
device.awgs[0].outputs["*"].modulation.mode(0)
device.awgs[0].time(0)
device.awgs[0].userregs(0)

#set DIO for internal triggering, check https://www.zhinst.com/others/en/blogs/synchronizing-multiple-awg-channels
device.dios[0].mode(1)
device.dios[0].drive(1)
device.awgs['*'].dio.valid.polarity(2)
device.awgs["*"].dio.valid.index(0)
device.awgs['*'].dio.strobe.slope(0)

#### Waveform generation

In [17]:



### Generate Rabi sequences

init_t = 1e-9
fin_t = 1500e-9
incr_t = 50e-9
con_freq = 200e6 # IF frequency

read_t = 1e-6
read_freq = 500e6
mark_len = 500              #in 2.4GHz rate
wait_n = 1000000               # in 3.3ns pts
delay_adjust = 1500

control_pulses_I = []
control_pulses_Q = []
con_pul_len = []
marker_pulses = []

sclk = 2.4e9 #sample clock
t_list = np.arange(init_t,fin_t,incr_t)

for i in t_list:
    pul = gen_sin(con_freq,0,1.0,i,sclk)
    pul = round_off_wvf_to_16_wvfpoints(pul,front=True)
    control_pulses_I.append(pul)

    pul = gen_sin(con_freq,np.pi/2,1.0,i,sclk)
    pul = round_off_wvf_to_16_wvfpoints(pul,front=True)
    control_pulses_Q.append(pul)


    con_pul_len.append(len(pul))
    mar = np.zeros_like(pul)
    mar[-30:-20] = 1
    marker_pulses.append(mar)

readout_pulse_I = round_off_wvf_to_16_wvfpoints(gen_sin(read_freq,0,1.0,read_t,sclk),back=True)
readout_pulse_Q = round_off_wvf_to_16_wvfpoints(gen_sin(read_freq,np.pi/2,1.0,read_t,sclk),back=True)

marker = np.append(np.ones(mark_len),np.zeros(readout_pulse_I.size-mark_len))


In [15]:
awg_program = Sequence()
awg_program.waveforms = Waveforms()
awg_program.waveforms_read = Waveforms()

#add control pulses
for i, (pul_I, pul_Q) in enumerate(zip(control_pulses_I,control_pulses_Q)):
    awg_program.waveforms.assign_waveform(i+1,
                                        Wave(pul_I, name= f"rabi_I_{i+1}"),
                                        Wave(pul_Q, name= f"rabi_Q_{i+1}"),
                                        markers=marker_pulses[i])

# add readout pulse
awg_program.waveforms.assign_waveform(0,
                                        Wave(readout_pulse_I, name= f"readout_I"),
                                        Wave(readout_pulse_Q, name= f"readout_Q"),
                                        markers=15*marker)

awg_program.waveforms_read.assign_waveform(0,
                                        Wave(readout_pulse_I, name= f"readout_I1"),
                                        Wave(readout_pulse_Q, name= f"readout_Q1"),
                                        markers=15*marker)

awg_program.code = """
while (true) {

"""

#add placeholders to the sequence
awg_program.code += awg_program.waveforms.get_sequence_snippet()

for i in range(len(control_pulses_I)):
    awg_program.code += f"\n playWave(1,rabi_I_{i+1},2,rabi_Q_{i+1});\n waitWave();\n playWave(3,readout_I,4,readout_Q);\n wait({wait_n});"

awg_program.code += '}'

In [16]:
comp_upload_seq(awgnode=0,code=awg_program.code,device=device,session=session)

Compilation started
Detected 1 devices with a total of 4 AWG cores.
Compiling source string for core 1 of 4
FIFO play mode is required on device type 'HDAWG'
Compilation failed


RuntimeError: Error during sequencer compilation. Check the log for detailed information

In [9]:
with device.set_transaction():
    # device.awgs[1].enable(False)
    device.awgs[0].enable(False)
    device.awgs[0].write_to_waveform_memory(awg_program.waveforms)
    print("AWG0 done")
    device.awgs[1].write_to_waveform_memory(awg_program.waveforms_read)
    print("AWG1 done")

AWG0 done
AWG1 done


In [10]:
device.awgs[0].enable(True)